# How many runs do you need? Run-to-run noise on ARC-AGI-3

Identical code does not give identical ARC-AGI-3 scores. Five byte-identical submissions of one of our agents scored with
an SD of **2.87 points** on the public leaderboard, and six runs of one agent on the 25 public games varied by
**5.3 levels** (SD; mean 102.5 levels). This notebook turns a noise level into four practical answers:

1. how many runs per arm a comparison needs,
2. how large a difference one run per arm can show,
3. how uncertain the noise estimate itself is, and
4. how much keeping the best of *n* identical resubmissions lifts a public score.

The numbers come from our ARC Prize 2026 paper (Shreekumar Shah). Change `SIGMA_LB` and `SIGMA_DEV`, or paste your own
scores in the last section.

In [ ]:
import math
from statistics import NormalDist
import numpy as np

SIGMA_LB = 2.87    # SD of one leaderboard submission, points (5 byte-identical submissions)
SIGMA_DEV = 5.3    # SD of one dev run on the 25 public games, levels (6 runs of one agent)
DEV_MEAN = 102.5   # mean levels of those 6 runs
ALPHA, POWER = 0.05, 0.80
Z = NormalDist().inv_cdf(1 - ALPHA / 2) + NormalDist().inv_cdf(POWER)

def runs_per_arm(sigma, delta):
    """Normal approximation: n = 2 (z_a + z_b)^2 sigma^2 / delta^2 runs in each arm."""
    return math.ceil(2 * Z * Z * sigma * sigma / (delta * delta))

def smallest_difference(sigma, n):
    """Smallest true difference that n runs per arm detect with the chosen power."""
    return Z * sigma * math.sqrt(2 / n)

## 1. Runs per arm (two-sided test at 0.05, power 0.8)

In [ ]:
print(f'Leaderboard, SD {SIGMA_LB} points per submission')
for d in (1, 2, 3, 5, 8):
    print(f'  true gain +{d} points: {runs_per_arm(SIGMA_LB, d):4d} submissions per arm')
print(f'Development runs on the 25 public games, SD {SIGMA_DEV} levels per run (mean {DEV_MEAN})')
for pct in (2, 5, 10, 20):
    d = DEV_MEAN * pct / 100
    print(f'  true gain +{pct}% levels (+{d:.1f}): {runs_per_arm(SIGMA_DEV, d):4d} runs per arm')

## 2. What a small comparison can show

In [ ]:
for n in (1, 2, 3, 5, 10):
    lb, dev = smallest_difference(SIGMA_LB, n), smallest_difference(SIGMA_DEV, n)
    print(f'{n:2d} run(s) per arm: leaderboard {lb:5.1f} points | dev {dev:5.1f} levels ({100 * dev / DEV_MEAN:.0f}%)')

## 3. With few runs, check the formula by simulation

The formula assumes the SD is known. In practice it is estimated from the same runs, so the test is a t-test and the
real power at small *n* is a little lower. A quick simulation finds the smallest *n* that reaches power 0.8.

In [ ]:
rng = np.random.default_rng(0)

def t_stats(sigma, delta, n, R):
    a, b = rng.normal(0, sigma, (R, n)), rng.normal(delta, sigma, (R, n))
    return (b.mean(1) - a.mean(1)) / np.sqrt((a.var(1, ddof=1) + b.var(1, ddof=1)) / n)

def t_power(sigma, delta, n, R=40000):
    crit = np.quantile(np.abs(t_stats(sigma, 0.0, n, R)), 1 - ALPHA)   # simulated critical value
    return float((np.abs(t_stats(sigma, delta, n, R)) > crit).mean())

for label, sigma, delta in (('leaderboard +3 points', SIGMA_LB, 3), ('leaderboard +5 points', SIGMA_LB, 5),
                            ('dev +10% levels', SIGMA_DEV, 0.10 * DEV_MEAN), ('dev +20% levels', SIGMA_DEV, 0.20 * DEV_MEAN)):
    n0 = max(2, runs_per_arm(sigma, delta)); n = n0
    while t_power(sigma, delta, n) < POWER:
        n += 1
    print(f'{label}: formula {n0} per arm (simulated power {t_power(sigma, delta, n0):.2f}); t-test needs {n}')

## 4. How sure are you of the noise?

An SD from five or six runs is itself loose. The 90% interval below uses the chi-square distribution of the sample
variance (normal runs assumed), computed exactly. Plan with the upper end if a wrong "no difference" would be costly.

In [ ]:
def p_lower(a, x):   # regularized lower incomplete gamma (series), for chi-square quantiles without scipy
    s = term = 1.0 / a; k = 0
    while abs(term) > 1e-15 * abs(s):
        k += 1; term *= x / (a + k); s += term
    return s * math.exp(-x + a * math.log(x) - math.lgamma(a))

def chi2_q(p, df):
    lo, hi = 1e-9, 200.0
    for _ in range(200):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if p_lower(df / 2, mid / 2) < p else (lo, mid)
    return (lo + hi) / 2

for label, s, k, d in (('leaderboard', SIGMA_LB, 5, 3.0), ('dev', SIGMA_DEV, 6, 0.10 * DEV_MEAN)):
    lo, hi = s * math.sqrt((k - 1) / chi2_q(0.95, k - 1)), s * math.sqrt((k - 1) / chi2_q(0.05, k - 1))
    print(f'{label}: SD {s} from {k} runs -> 90% interval {lo:.2f} to {hi:.2f}; '
          f'runs per arm for +{d:g}: {runs_per_arm(lo, d)} to {runs_per_arm(hi, d)} (point estimate {runs_per_arm(s, d)})')

## 5. Best of *n* identical resubmissions

If a team resubmits identical code and keeps its best public score, the expected best is the mean plus
SD x e_n, where e_n is the expected maximum of n standard normal draws (computed exactly below).

In [ ]:
def e_max(n):
    x = np.linspace(-12, 12, 240001); phi = np.exp(-x * x / 2) / math.sqrt(2 * math.pi)
    Phi = 0.5 * (1 + np.vectorize(math.erf)(x / math.sqrt(2)))
    return float(np.sum(x * n * phi * Phi ** (n - 1)) * (x[1] - x[0]))

for n in (1, 2, 5, 10, 20, 50, 100, 200):
    e = e_max(n)
    sim = rng.standard_normal((20000, n)).max(1).mean()
    print(f'best of {n:3d}: +{SIGMA_LB * e:.2f} points above the mean (e_n {e:.3f}; simulation {sim:.3f})')

## 6. Your own numbers

Paste the public scores of identical resubmissions of your own code (at least two).

In [ ]:
my_scores = []   # for example [24.1, 27.3, 22.8, 25.0]
if len(my_scores) >= 2:
    s = float(np.std(my_scores, ddof=1))
    print(f'your SD from {len(my_scores)} submissions: {s:.2f} points')
    for d in (2, 3, 5):
        print(f'  +{d} points: {runs_per_arm(s, d)} submissions per arm')
    print(f'  best of 5 adds about {s * e_max(5):.2f} points to your mean')
else:
    print('add at least two scores to my_scores')

## 7. Compare two agents on crack cost (two numbers)

A level total mixes two things: how many tokens an agent generates (supply S) and how many tokens it needs per crack
(crack cost b). On the **same levels**, b can be estimated far more precisely than a level total, because each level's
own difficulty is fitted and removed: log T(run, level) = a_level + b_run + noise, with attempts still open at the end
counted as censored (a lognormal accelerated-failure-time model, fitted by EM). b is a log multiplier: b = +0.1 means
about 10% more tokens per crack than the reference arm.

The example compares dfranzen's Milestone 2 notebook (B1, two 25-minute runs on 10 public games) with our B12 (seven
121-minute runs) on the levels of those 10 games. Replace `ROWS`, `ARM_A` and `ARM_B` with your own attempts: one row per
level attempt, `[run, game, level, generated tokens, 1 if cleared else 0]`.

In [ ]:
ARM_A = ["b1_franzen", "b1c_ctrl"]
ARM_B = ["b12_fb25", "b28_ctl_fb25", "b32_ctl_r2", "b32_ctl_r3", "b32_spec2_r2", "b32_spec2_r3", "t1c_fb25_ctl"]
ROWS = [["b1_franzen","tu93",0,6741,1],["b1_franzen","tu93",1,39655,1],["b1_franzen","tu93",2,11895,1],["b1_franzen","tu93",3,15210,0],["b1_franzen","lp85",0,5813,1],["b1_franzen","lp85",1,16813,1],["b1_franzen","lp85",2,18931,1],["b1_franzen","lp85",3,7894,1],["b1_franzen","lp85",4,8321,1],["b1_franzen","lp85",5,40299,0],["b1_franzen","vc33",0,12674,1],["b1_franzen","vc33",1,9406,1],["b1_franzen","vc33",2,27766,1],["b1_franzen","vc33",3,44264,0],["b1_franzen","r11l",0,10572,1],["b1_franzen","r11l",1,34261,1],["b1_franzen","r11l",2,6052,1],["b1_franzen","r11l",3,8987,1],["b1_franzen","r11l",4,12467,1],["b1_franzen","r11l",5,21163,0],["b1_franzen","sc25",0,49074,1],["b1_franzen","sc25",1,2797,1],["b1_franzen","sc25",2,12288,1],["b1_franzen","sc25",3,22819,1],["b1_franzen","ar25",0,6191,1],["b1_franzen","ar25",1,15353,1],["b1_franzen","ar25",2,14580,1],["b1_franzen","ar25",3,9400,1],["b1_franzen","ar25",4,39512,0],["b1_franzen","sb26",0,5229,1],["b1_franzen","sb26",1,32823,1],["b1_franzen","sb26",2,3621,1],["b1_franzen","sb26",3,2712,1],["b1_franzen","sb26",4,2626,1],["b1_franzen","sb26",5,2378,1],["b1_franzen","sb26",6,9430,1],["b1_franzen","sb26",7,11418,1],["b1_franzen","re86",0,6994,1],["b1_franzen","re86",1,18178,1],["b1_franzen","re86",2,8127,1],["b1_franzen","re86",3,18497,1],["b1_franzen","re86",4,30659,0],["b1_franzen","ft09",0,5501,1],["b1_franzen","ft09",1,4057,1],["b1_franzen","ft09",2,9713,1],["b1_franzen","ft09",3,11159,1],["b1_franzen","ft09",4,44801,0],["b1_franzen","tr87",0,43700,1],["b1_franzen","tr87",1,15020,1],["b1_franzen","tr87",2,10428,1],["b1_franzen","tr87",3,5681,1],["b1_franzen","tr87",4,5343,0],["b1c_ctrl","tu93",0,23723,1],["b1c_ctrl","tu93",1,23968,1],["b1c_ctrl","tu93",2,14853,1],["b1c_ctrl","tu93",3,36346,0],["b1c_ctrl","lp85",0,6752,1],["b1c_ctrl","lp85",1,29219,1],["b1c_ctrl","lp85",2,7903,1],["b1c_ctrl","lp85",3,16737,1],["b1c_ctrl","lp85",4,11797,1],["b1c_ctrl","lp85",5,5081,0],["b1c_ctrl","vc33",0,8786,1],["b1c_ctrl","vc33",1,9970,1],["b1c_ctrl","vc33",2,16217,1],["b1c_ctrl","vc33",3,61485,0],["b1c_ctrl","r11l",0,18718,1],["b1c_ctrl","r11l",1,47638,1],["b1c_ctrl","sc25",0,14243,1],["b1c_ctrl","sc25",1,5054,1],["b1c_ctrl","sc25",2,58887,1],["b1c_ctrl","sc25",3,14273,0],["b1c_ctrl","ar25",0,6292,1],["b1c_ctrl","ar25",1,15930,1],["b1c_ctrl","ar25",2,21401,1],["b1c_ctrl","ar25",3,8226,1],["b1c_ctrl","ar25",4,29143,1],["b1c_ctrl","ar25",5,8002,0],["b1c_ctrl","sb26",0,2201,1],["b1c_ctrl","sb26",1,18859,1],["b1c_ctrl","sb26",2,3116,1],["b1c_ctrl","sb26",3,2037,1],["b1c_ctrl","sb26",4,4487,1],["b1c_ctrl","sb26",5,2440,1],["b1c_ctrl","sb26",6,17696,1],["b1c_ctrl","sb26",7,18812,1],["b1c_ctrl","re86",0,10497,1],["b1c_ctrl","re86",1,11007,1],["b1c_ctrl","re86",2,28629,1],["b1c_ctrl","re86",3,22068,1],["b1c_ctrl","re86",4,7609,0],["b1c_ctrl","ft09",0,6025,1],["b1c_ctrl","ft09",1,3397,1],["b1c_ctrl","ft09",2,8534,1],["b1c_ctrl","ft09",3,7022,1],["b1c_ctrl","ft09",4,24830,1],["b1c_ctrl","ft09",5,12076,1],["b1c_ctrl","tr87",0,45974,1],["b1c_ctrl","tr87",1,8648,1],["b1c_ctrl","tr87",2,5530,1],["b1c_ctrl","tr87",3,5194,1],["b1c_ctrl","tr87",4,28127,0],["b12_fb25","tu93",0,13195,1],["b12_fb25","tu93",1,46733,1],["b12_fb25","tu93",2,16564,1],["b12_fb25","tu93",3,67906,1],["b12_fb25","tu93",4,49262,1],["b12_fb25","tu93",5,56001,1],["b12_fb25","tu93",6,23893,1],["b12_fb25","tu93",7,35630,1],["b12_fb25","tu93",8,63159,0],["b12_fb25","lp85",0,7580,1],["b12_fb25","lp85",1,22807,1],["b12_fb25","lp85",2,12053,1],["b12_fb25","lp85",3,33347,1],["b12_fb25","lp85",4,12024,1],["b12_fb25","lp85",5,24141,1],["b12_fb25","lp85",6,9601,1],["b12_fb25","lp85",7,52515,1],["b12_fb25","vc33",0,8743,1],["b12_fb25","vc33",1,8900,1],["b12_fb25","vc33",2,25099,1],["b12_fb25","vc33",3,50084,1],["b12_fb25","vc33",4,52552,1],["b12_fb25","vc33",5,19825,1],["b12_fb25","vc33",6,33094,1],["b12_fb25","r11l",0,30878,1],["b12_fb25","r11l",1,69611,1],["b12_fb25","r11l",2,19282,1],["b12_fb25","r11l",3,21578,1],["b12_fb25","r11l",4,42992,1],["b12_fb25","r11l",5,65262,1],["b12_fb25","sc25",0,22990,1],["b12_fb25","sc25",1,7030,1],["b12_fb25","sc25",2,10106,1],["b12_fb25","sc25",3,61130,1],["b12_fb25","sc25",4,10795,0],["b12_fb25","ar25",0,5205,1],["b12_fb25","ar25",1,7844,1],["b12_fb25","ar25",2,25381,1],["b12_fb25","ar25",3,13498,1],["b12_fb25","ar25",4,35582,1],["b12_fb25","ar25",5,33182,1],["b12_fb25","ar25",6,7286,1],["b12_fb25","ar25",7,6360,1],["b12_fb25","sb26",0,2932,1],["b12_fb25","sb26",1,31707,1],["b12_fb25","sb26",2,2468,1],["b12_fb25","sb26",3,2010,1],["b12_fb25","sb26",4,5498,1],["b12_fb25","sb26",5,2268,1],["b12_fb25","sb26",6,10960,1],["b12_fb25","sb26",7,25458,1],["b12_fb25","re86",0,11006,1],["b12_fb25","re86",1,7944,1],["b12_fb25","re86",2,17644,1],["b12_fb25","re86",3,28656,1],["b12_fb25","re86",4,47676,1],["b12_fb25","re86",5,84994,0],["b12_fb25","ft09",0,8302,1],["b12_fb25","ft09",1,6715,1],["b12_fb25","ft09",2,6102,1],["b12_fb25","ft09",3,9047,1],["b12_fb25","ft09",4,16006,1],["b12_fb25","ft09",5,12404,1],["b12_fb25","tr87",0,15756,1],["b12_fb25","tr87",1,12960,1],["b12_fb25","tr87",2,4208,1],["b12_fb25","tr87",3,2206,1],["b12_fb25","tr87",4,47000,1],["b12_fb25","tr87",5,42227,1],["b28_ctl_fb25","tu93",0,10360,1],["b28_ctl_fb25","tu93",1,12080,1],["b28_ctl_fb25","tu93",2,19261,1],["b28_ctl_fb25","tu93",3,34593,1],["b28_ctl_fb25","tu93",4,12495,1],["b28_ctl_fb25","tu93",5,45466,1],["b28_ctl_fb25","tu93",6,10549,1],["b28_ctl_fb25","tu93",7,63653,1],["b28_ctl_fb25","tu93",8,58019,0],["b28_ctl_fb25","lp85",0,14111,1],["b28_ctl_fb25","lp85",1,33521,1],["b28_ctl_fb25","lp85",2,8630,1],["b28_ctl_fb25","lp85",3,15650,1],["b28_ctl_fb25","lp85",4,20545,1],["b28_ctl_fb25","lp85",5,41442,1],["b28_ctl_fb25","lp85",6,7598,1],["b28_ctl_fb25","lp85",7,26639,1],["b28_ctl_fb25","vc33",0,8172,1],["b28_ctl_fb25","vc33",1,8333,1],["b28_ctl_fb25","vc33",2,39556,1],["b28_ctl_fb25","vc33",3,60469,1],["b28_ctl_fb25","vc33",4,39635,1],["b28_ctl_fb25","vc33",5,27490,1],["b28_ctl_fb25","vc33",6,22385,1],["b28_ctl_fb25","r11l",0,8022,1],["b28_ctl_fb25","r11l",1,57651,1],["b28_ctl_fb25","r11l",2,51183,1],["b28_ctl_fb25","r11l",3,61861,1],["b28_ctl_fb25","r11l",4,39500,1],["b28_ctl_fb25","r11l",5,47501,1],["b28_ctl_fb25","sc25",0,100474,0],["b28_ctl_fb25","ar25",0,4090,1],["b28_ctl_fb25","ar25",1,11740,1],["b28_ctl_fb25","ar25",2,9393,1],["b28_ctl_fb25","ar25",3,10731,1],["b28_ctl_fb25","ar25",4,26593,1],["b28_ctl_fb25","ar25",5,55636,1],["b28_ctl_fb25","ar25",6,15899,1],["b28_ctl_fb25","ar25",7,6235,1],["b28_ctl_fb25","sb26",0,3766,1],["b28_ctl_fb25","sb26",1,5058,1],["b28_ctl_fb25","sb26",2,1637,1],["b28_ctl_fb25","sb26",3,3056,1],["b28_ctl_fb25","sb26",4,5958,1],["b28_ctl_fb25","sb26",5,2210,1],["b28_ctl_fb25","sb26",6,3385,1],["b28_ctl_fb25","sb26",7,61339,1],["b28_ctl_fb25","re86",0,3430,1],["b28_ctl_fb25","re86",1,11284,1],["b28_ctl_fb25","re86",2,7730,1],["b28_ctl_fb25","re86",3,14095,1],["b28_ctl_fb25","re86",4,28307,1],["b28_ctl_fb25","re86",5,113241,0],["b28_ctl_fb25","ft09",0,11951,1],["b28_ctl_fb25","ft09",1,3418,1],["b28_ctl_fb25","ft09",2,6857,1],["b28_ctl_fb25","ft09",3,10523,1],["b28_ctl_fb25","ft09",4,167037,0],["b28_ctl_fb25","tr87",0,32059,1],["b28_ctl_fb25","tr87",1,10093,1],["b28_ctl_fb25","tr87",2,6175,1],["b28_ctl_fb25","tr87",3,10976,1],["b28_ctl_fb25","tr87",4,63525,1],["b28_ctl_fb25","tr87",5,36851,1],["b32_ctl_r2","tu93",0,10462,1],["b32_ctl_r2","tu93",1,28599,1],["b32_ctl_r2","tu93",2,19878,1],["b32_ctl_r2","tu93",3,69073,1],["b32_ctl_r2","tu93",4,21850,1],["b32_ctl_r2","tu93",5,35290,1],["b32_ctl_r2","tu93",6,24906,1],["b32_ctl_r2","tu93",7,70097,1],["b32_ctl_r2","tu93",8,35298,0],["b32_ctl_r2","lp85",0,14811,1],["b32_ctl_r2","lp85",1,17215,1],["b32_ctl_r2","lp85",2,12545,1],["b32_ctl_r2","lp85",3,17378,1],["b32_ctl_r2","lp85",4,11659,1],["b32_ctl_r2","lp85",5,23862,1],["b32_ctl_r2","lp85",6,10114,1],["b32_ctl_r2","lp85",7,49644,1],["b32_ctl_r2","vc33",0,6529,1],["b32_ctl_r2","vc33",1,7387,1],["b32_ctl_r2","vc33",2,33995,1],["b32_ctl_r2","vc33",3,123129,0],["b32_ctl_r2","r11l",0,14559,1],["b32_ctl_r2","r11l",1,40916,1],["b32_ctl_r2","r11l",2,29102,1],["b32_ctl_r2","r11l",3,61830,1],["b32_ctl_r2","r11l",4,71515,1],["b32_ctl_r2","r11l",5,47341,1],["b32_ctl_r2","sc25",0,20450,1],["b32_ctl_r2","sc25",1,6039,1],["b32_ctl_r2","sc25",2,19874,1],["b32_ctl_r2","sc25",3,64910,1],["b32_ctl_r2","sc25",4,69013,1],["b32_ctl_r2","sc25",5,21935,1],["b32_ctl_r2","ar25",0,5176,1],["b32_ctl_r2","ar25",1,29533,1],["b32_ctl_r2","ar25",2,12096,1],["b32_ctl_r2","ar25",3,8096,1],["b32_ctl_r2","ar25",4,36103,1],["b32_ctl_r2","ar25",5,19551,1],["b32_ctl_r2","ar25",6,15549,1],["b32_ctl_r2","ar25",7,25030,1],["b32_ctl_r2","sb26",0,2461,1],["b32_ctl_r2","sb26",1,13112,1],["b32_ctl_r2","sb26",2,4017,1],["b32_ctl_r2","sb26",3,2246,1],["b32_ctl_r2","sb26",4,2509,1],["b32_ctl_r2","sb26",5,2106,1],["b32_ctl_r2","sb26",6,4514,1],["b32_ctl_r2","sb26",7,5708,1],["b32_ctl_r2","re86",0,6122,1],["b32_ctl_r2","re86",1,19371,1],["b32_ctl_r2","re86",2,7403,1],["b32_ctl_r2","re86",3,21480,1],["b32_ctl_r2","re86",4,35655,1],["b32_ctl_r2","re86",5,101920,1],["b32_ctl_r2","re86",6,18995,0],["b32_ctl_r2","ft09",0,7722,1],["b32_ctl_r2","ft09",1,4816,1],["b32_ctl_r2","ft09",2,7137,1],["b32_ctl_r2","ft09",3,13741,1],["b32_ctl_r2","ft09",4,53044,1],["b32_ctl_r2","ft09",5,8014,1],["b32_ctl_r2","tr87",0,23122,1],["b32_ctl_r2","tr87",1,16946,1],["b32_ctl_r2","tr87",2,10630,1],["b32_ctl_r2","tr87",3,3749,1],["b32_ctl_r2","tr87",4,10025,1],["b32_ctl_r2","tr87",5,26668,1],["b32_ctl_r3","tu93",0,7823,1],["b32_ctl_r3","tu93",1,16770,1],["b32_ctl_r3","tu93",2,12642,1],["b32_ctl_r3","tu93",3,24488,1],["b32_ctl_r3","tu93",4,14734,1],["b32_ctl_r3","tu93",5,32900,1],["b32_ctl_r3","tu93",6,8169,1],["b32_ctl_r3","tu93",7,48122,1],["b32_ctl_r3","tu93",8,57265,0],["b32_ctl_r3","lp85",0,9749,1],["b32_ctl_r3","lp85",1,26131,1],["b32_ctl_r3","lp85",2,14687,1],["b32_ctl_r3","lp85",3,32896,1],["b32_ctl_r3","lp85",4,21551,1],["b32_ctl_r3","lp85",5,18667,1],["b32_ctl_r3","lp85",6,8645,1],["b32_ctl_r3","lp85",7,31552,1],["b32_ctl_r3","vc33",0,6969,1],["b32_ctl_r3","vc33",1,7129,1],["b32_ctl_r3","vc33",2,18900,1],["b32_ctl_r3","vc33",3,41564,1],["b32_ctl_r3","vc33",4,30362,1],["b32_ctl_r3","vc33",5,14032,1],["b32_ctl_r3","vc33",6,22425,1],["b32_ctl_r3","r11l",0,40826,1],["b32_ctl_r3","r11l",1,72737,1],["b32_ctl_r3","r11l",2,95747,1],["b32_ctl_r3","r11l",3,56481,1],["b32_ctl_r3","r11l",4,25515,1],["b32_ctl_r3","r11l",5,37498,1],["b32_ctl_r3","sc25",0,47516,1],["b32_ctl_r3","sc25",1,5856,1],["b32_ctl_r3","sc25",2,45053,1],["b32_ctl_r3","sc25",3,17479,1],["b32_ctl_r3","sc25",4,76629,0],["b32_ctl_r3","ar25",0,8880,1],["b32_ctl_r3","ar25",1,7380,1],["b32_ctl_r3","ar25",2,16687,1],["b32_ctl_r3","ar25",3,9535,1],["b32_ctl_r3","ar25",4,17806,1],["b32_ctl_r3","ar25",5,12488,1],["b32_ctl_r3","ar25",6,43916,1],["b32_ctl_r3","ar25",7,8272,1],["b32_ctl_r3","sb26",0,1537,1],["b32_ctl_r3","sb26",1,42400,1],["b32_ctl_r3","sb26",2,3134,1],["b32_ctl_r3","sb26",3,5635,1],["b32_ctl_r3","sb26",4,2424,1],["b32_ctl_r3","sb26",5,2209,1],["b32_ctl_r3","sb26",6,13106,1],["b32_ctl_r3","sb26",7,4339,1],["b32_ctl_r3","re86",0,10046,1],["b32_ctl_r3","re86",1,11265,1],["b32_ctl_r3","re86",2,11403,1],["b32_ctl_r3","re86",3,32692,1],["b32_ctl_r3","re86",4,34853,1],["b32_ctl_r3","re86",5,89377,1],["b32_ctl_r3","re86",6,23592,0],["b32_ctl_r3","ft09",0,5925,1],["b32_ctl_r3","ft09",1,20908,1],["b32_ctl_r3","ft09",2,16064,1],["b32_ctl_r3","ft09",3,11446,1],["b32_ctl_r3","ft09",4,86933,1],["b32_ctl_r3","ft09",5,15245,1],["b32_ctl_r3","tr87",0,27484,1],["b32_ctl_r3","tr87",1,4585,1],["b32_ctl_r3","tr87",2,3644,1],["b32_ctl_r3","tr87",3,2844,1],["b32_ctl_r3","tr87",4,12433,1],["b32_ctl_r3","tr87",5,38691,1],["b32_spec2_r2","tu93",0,11543,1],["b32_spec2_r2","tu93",1,108055,1],["b32_spec2_r2","tu93",2,46865,1],["b32_spec2_r2","tu93",3,24344,1],["b32_spec2_r2","tu93",4,32751,1],["b32_spec2_r2","tu93",5,28000,0],["b32_spec2_r2","lp85",0,9259,1],["b32_spec2_r2","lp85",1,32777,1],["b32_spec2_r2","lp85",2,16333,1],["b32_spec2_r2","lp85",3,12957,1],["b32_spec2_r2","lp85",4,14774,1],["b32_spec2_r2","lp85",5,28686,1],["b32_spec2_r2","lp85",6,10814,1],["b32_spec2_r2","lp85",7,32505,1],["b32_spec2_r2","vc33",0,13421,1],["b32_spec2_r2","vc33",1,8669,1],["b32_spec2_r2","vc33",2,12962,1],["b32_spec2_r2","vc33",3,98808,1],["b32_spec2_r2","vc33",4,39382,1],["b32_spec2_r2","vc33",5,12584,1],["b32_spec2_r2","vc33",6,27542,1],["b32_spec2_r2","r11l",0,15593,1],["b32_spec2_r2","r11l",1,49621,1],["b32_spec2_r2","r11l",2,29749,1],["b32_spec2_r2","r11l",3,24547,1],["b32_spec2_r2","r11l",4,76031,1],["b32_spec2_r2","r11l",5,23673,1],["b32_spec2_r2","sc25",0,38047,1],["b32_spec2_r2","sc25",1,4291,1],["b32_spec2_r2","sc25",2,46199,1],["b32_spec2_r2","sc25",3,27989,1],["b32_spec2_r2","sc25",4,75282,0],["b32_spec2_r2","ar25",0,4525,1],["b32_spec2_r2","ar25",1,12082,1],["b32_spec2_r2","ar25",2,22238,1],["b32_spec2_r2","ar25",3,5962,1],["b32_spec2_r2","ar25",4,28046,1],["b32_spec2_r2","ar25",5,20890,1],["b32_spec2_r2","ar25",6,23105,1],["b32_spec2_r2","ar25",7,22022,1],["b32_spec2_r2","sb26",0,7114,1],["b32_spec2_r2","sb26",1,12187,1],["b32_spec2_r2","sb26",2,2245,1],["b32_spec2_r2","sb26",3,3172,1],["b32_spec2_r2","sb26",4,4290,1],["b32_spec2_r2","sb26",5,2040,1],["b32_spec2_r2","sb26",6,3589,1],["b32_spec2_r2","sb26",7,21277,1],["b32_spec2_r2","re86",0,5164,1],["b32_spec2_r2","re86",1,37794,1],["b32_spec2_r2","re86",2,10485,1],["b32_spec2_r2","re86",3,13451,1],["b32_spec2_r2","re86",4,36442,1],["b32_spec2_r2","re86",5,78416,1],["b32_spec2_r2","re86",6,60316,0],["b32_spec2_r2","ft09",0,9867,1],["b32_spec2_r2","ft09",1,2899,1],["b32_spec2_r2","ft09",2,9704,1],["b32_spec2_r2","ft09",3,9146,1],["b32_spec2_r2","ft09",4,42114,1],["b32_spec2_r2","ft09",5,10090,1],["b32_spec2_r2","tr87",0,29820,1],["b32_spec2_r2","tr87",1,14083,1],["b32_spec2_r2","tr87",2,7113,1],["b32_spec2_r2","tr87",3,3481,1],["b32_spec2_r2","tr87",4,27824,1],["b32_spec2_r2","tr87",5,32953,1],["b32_spec2_r3","tu93",0,8445,1],["b32_spec2_r3","tu93",1,49408,1],["b32_spec2_r3","tu93",2,22358,1],["b32_spec2_r3","tu93",3,22762,1],["b32_spec2_r3","tu93",4,13710,1],["b32_spec2_r3","tu93",5,20275,1],["b32_spec2_r3","tu93",6,30755,1],["b32_spec2_r3","tu93",7,25834,1],["b32_spec2_r3","tu93",8,30661,1],["b32_spec2_r3","lp85",0,12246,1],["b32_spec2_r3","lp85",1,16742,1],["b32_spec2_r3","lp85",2,10281,1],["b32_spec2_r3","lp85",3,9083,1],["b32_spec2_r3","lp85",4,14114,1],["b32_spec2_r3","lp85",5,28071,1],["b32_spec2_r3","lp85",6,6856,1],["b32_spec2_r3","lp85",7,18997,1],["b32_spec2_r3","vc33",0,12760,1],["b32_spec2_r3","vc33",1,13444,1],["b32_spec2_r3","vc33",2,20338,1],["b32_spec2_r3","vc33",3,51948,1],["b32_spec2_r3","vc33",4,68560,1],["b32_spec2_r3","vc33",5,11579,1],["b32_spec2_r3","vc33",6,80991,0],["b32_spec2_r3","r11l",0,10707,1],["b32_spec2_r3","r11l",1,44735,1],["b32_spec2_r3","r11l",2,31149,1],["b32_spec2_r3","r11l",3,16737,1],["b32_spec2_r3","r11l",4,113822,1],["b32_spec2_r3","sc25",0,62727,0],["b32_spec2_r3","ar25",0,5498,1],["b32_spec2_r3","ar25",1,12477,1],["b32_spec2_r3","ar25",2,18052,1],["b32_spec2_r3","ar25",3,5544,1],["b32_spec2_r3","ar25",4,29936,1],["b32_spec2_r3","ar25",5,13131,1],["b32_spec2_r3","ar25",6,17480,1],["b32_spec2_r3","ar25",7,39527,1],["b32_spec2_r3","sb26",0,3000,1],["b32_spec2_r3","sb26",1,4818,1],["b32_spec2_r3","sb26",2,1212,1],["b32_spec2_r3","sb26",3,3055,1],["b32_spec2_r3","sb26",4,2045,1],["b32_spec2_r3","sb26",5,1481,1],["b32_spec2_r3","sb26",6,6634,1],["b32_spec2_r3","sb26",7,6477,1],["b32_spec2_r3","re86",0,9858,1],["b32_spec2_r3","re86",1,10379,1],["b32_spec2_r3","re86",2,8190,1],["b32_spec2_r3","re86",3,19864,1],["b32_spec2_r3","re86",4,31331,1],["b32_spec2_r3","re86",5,64050,1],["b32_spec2_r3","re86",6,57066,1],["b32_spec2_r3","ft09",0,4581,1],["b32_spec2_r3","ft09",1,5187,1],["b32_spec2_r3","ft09",2,9747,1],["b32_spec2_r3","ft09",3,14149,1],["b32_spec2_r3","ft09",4,15881,1],["b32_spec2_r3","ft09",5,10063,1],["b32_spec2_r3","tr87",0,40993,1],["b32_spec2_r3","tr87",1,9779,1],["b32_spec2_r3","tr87",2,4371,1],["b32_spec2_r3","tr87",3,3331,1],["b32_spec2_r3","tr87",4,30941,1],["b32_spec2_r3","tr87",5,43819,1],["t1c_fb25_ctl","tu93",0,4542,1],["t1c_fb25_ctl","tu93",1,9332,1],["t1c_fb25_ctl","tu93",2,9574,1],["t1c_fb25_ctl","tu93",3,40473,1],["t1c_fb25_ctl","tu93",4,50487,1],["t1c_fb25_ctl","tu93",5,68452,1],["t1c_fb25_ctl","tu93",6,11155,1],["t1c_fb25_ctl","tu93",7,18551,1],["t1c_fb25_ctl","tu93",8,17723,0],["t1c_fb25_ctl","lp85",0,6260,1],["t1c_fb25_ctl","lp85",1,30422,1],["t1c_fb25_ctl","lp85",2,22384,1],["t1c_fb25_ctl","lp85",3,24970,1],["t1c_fb25_ctl","lp85",4,11883,1],["t1c_fb25_ctl","lp85",5,52360,1],["t1c_fb25_ctl","lp85",6,8105,1],["t1c_fb25_ctl","lp85",7,26940,1],["t1c_fb25_ctl","vc33",0,16057,1],["t1c_fb25_ctl","vc33",1,6833,1],["t1c_fb25_ctl","vc33",2,19205,1],["t1c_fb25_ctl","vc33",3,135338,1],["t1c_fb25_ctl","vc33",4,26002,1],["t1c_fb25_ctl","vc33",5,26623,1],["t1c_fb25_ctl","vc33",6,38706,0],["t1c_fb25_ctl","r11l",0,20293,1],["t1c_fb25_ctl","r11l",1,56166,1],["t1c_fb25_ctl","r11l",2,29200,1],["t1c_fb25_ctl","r11l",3,26550,1],["t1c_fb25_ctl","r11l",4,31439,1],["t1c_fb25_ctl","r11l",5,20675,1],["t1c_fb25_ctl","sc25",0,39705,1],["t1c_fb25_ctl","sc25",1,3853,1],["t1c_fb25_ctl","sc25",2,47751,1],["t1c_fb25_ctl","sc25",3,43244,1],["t1c_fb25_ctl","sc25",4,52128,1],["t1c_fb25_ctl","sc25",5,49476,1],["t1c_fb25_ctl","ar25",0,7247,1],["t1c_fb25_ctl","ar25",1,15889,1],["t1c_fb25_ctl","ar25",2,11780,1],["t1c_fb25_ctl","ar25",3,7016,1],["t1c_fb25_ctl","ar25",4,17836,1],["t1c_fb25_ctl","ar25",5,12353,1],["t1c_fb25_ctl","ar25",6,38101,1],["t1c_fb25_ctl","ar25",7,7683,1],["t1c_fb25_ctl","sb26",0,3923,1],["t1c_fb25_ctl","sb26",1,24439,1],["t1c_fb25_ctl","sb26",2,2246,1],["t1c_fb25_ctl","sb26",3,2801,1],["t1c_fb25_ctl","sb26",4,21224,1],["t1c_fb25_ctl","sb26",5,2484,1],["t1c_fb25_ctl","sb26",6,26047,1],["t1c_fb25_ctl","sb26",7,3730,1],["t1c_fb25_ctl","re86",0,6064,1],["t1c_fb25_ctl","re86",1,10056,1],["t1c_fb25_ctl","re86",2,10257,1],["t1c_fb25_ctl","re86",3,9630,1],["t1c_fb25_ctl","re86",4,23606,1],["t1c_fb25_ctl","re86",5,73563,0],["t1c_fb25_ctl","ft09",0,6959,1],["t1c_fb25_ctl","ft09",1,2975,1],["t1c_fb25_ctl","ft09",2,11476,1],["t1c_fb25_ctl","ft09",3,8023,1],["t1c_fb25_ctl","ft09",4,26541,1],["t1c_fb25_ctl","ft09",5,18676,1],["t1c_fb25_ctl","tr87",0,30269,1],["t1c_fb25_ctl","tr87",1,16026,1],["t1c_fb25_ctl","tr87",2,5431,1],["t1c_fb25_ctl","tr87",3,5145,1],["t1c_fb25_ctl","tr87",4,15918,1],["t1c_fb25_ctl","tr87",5,36440,1]]

In [ ]:
def fit_aft(rows, ref, iters=200):
    runs = sorted({r[0] for r in rows}); levels = sorted({(r[1], r[2]) for r in rows})
    ri = {r: i for i, r in enumerate(runs)}; li = {l: i for i, l in enumerate(levels)}
    R = np.array([ri[r[0]] for r in rows]); L = np.array([li[(r[1], r[2])] for r in rows])
    y = np.log(np.maximum([r[3] for r in rows], 1)); c = np.array([r[4] for r in rows], bool)
    keep = np.isin(L, np.unique(L[c]))            # levels never cleared carry no information on b
    R, L, y, c = R[keep], L[keep], y[keep], c[keep]
    a, b, s = np.zeros(len(levels)), np.zeros(len(runs)), 1.0
    Phi = np.vectorize(lambda z: 0.5 * math.erfc(-z / math.sqrt(2)))
    for _ in range(iters):
        mu = a[L] + b[R]; z = (y - mu) / s
        lam = np.exp(-0.5 * z * z) / math.sqrt(2 * math.pi) / np.maximum(1 - Phi(z), 1e-12)
        ey = np.where(c, y, mu + s * lam); ev = np.where(c, (y - mu) ** 2, s * s * (1 + z * lam))
        for _ in range(3):
            a = np.bincount(L, ey - b[R], len(levels)) / np.maximum(np.bincount(L, minlength=len(levels)), 1)
            b = np.bincount(R, ey - a[L], len(runs)) / np.maximum(np.bincount(R, minlength=len(runs)), 1)
            b -= np.mean([b[ri[r]] for r in ref])
        mun = a[L] + b[R]; s = math.sqrt(np.mean(ev + 2 * (mu - mun) * (ey - mu) + (mu - mun) ** 2))
    return {r: float(b[ri[r]]) for r in runs}, s

b, s = fit_aft(ROWS, ARM_A)
d = np.mean([b[r] for r in ARM_B])
print(f'residual s {s:.2f}; b per run: ' + ', '.join(f'{r} {b[r]:+.3f}' for r in ARM_A + ARM_B))
print(f'arm B - arm A: b {d:+.3f}, so arm B needs x{math.exp(d):.2f} the tokens per crack')
rng = np.random.default_rng(0); boot = []
by_run = {r: [x for x in ROWS if x[0] == r] for r in ARM_A + ARM_B}
for _ in range(100):                               # bootstrap over runs within each arm (a refit each time)
    pa = [ARM_A[i] for i in rng.integers(0, len(ARM_A), len(ARM_A))]
    pb = [ARM_B[i] for i in rng.integers(0, len(ARM_B), len(ARM_B))]
    names = [f'{r}#{j}' for j, r in enumerate(pa + pb)]
    rows = [[n] + x[1:] for n, r in zip(names, pa + pb) for x in by_run[r]]
    bb, _ = fit_aft(rows, names[:len(pa)], iters=60)
    boot.append(np.mean([bb[n] for n in names[len(pa):]]))
print(f'bootstrap 90% interval for the difference: {np.percentile(boot, 5):+.3f} to {np.percentile(boot, 95):+.3f}')

## 8. Fit half, predict half: check the crack model on your own runs

Before trusting the model for an agent, check that it forecasts. Cut each run at half its clock, fit the three numbers
(wall share q, median wait, spread s) on the attempts seen so far, then predict each run's final level total. The second
half is assumed to generate as many tokens as the first, shared among the games still playing in proportion to their
first-half tokens. Levels are played in order; an attempt open at the cut continues from the tokens it already has. This
is the check of Section 4.2 of the paper (there: 101.1 levels predicted on average, 102.75 observed).

The example uses our four control runs: `GAMES[run]` lists, per game, its name, its number of levels, its attempts up to
the cut as `[generated tokens, 1 if cleared]`, and its generated tokens up to the cut; `OBSERVED[run]` is the final
level total.

In [ ]:
GAMES = {"b12_fb25":[["sk48",8,[[65328,1],[14507,0]],79835],["tn36",7,[[24351,1],[69068,0]],93419],["m0r0",6,[[25125,1],[24243,1],[91186,1],[16197,1],[9203,0]],165954],["bp35",9,[[39377,1],[13947,0]],53324],["cn04",6,[[38628,1],[46835,1],[85918,1],[18736,1],[21157,0]],211274],["dc22",6,[[25007,1],[23850,1],[108166,1],[46007,1],[14279,0]],217309],["tu93",9,[[13195,1],[46733,1],[16564,1],[67906,1],[31735,0]],176133],["lp85",8,[[7580,1],[22807,1],[12053,1],[33347,1],[12024,1],[24141,1],[0,0]],111952],["ka59",7,[[12127,1],[46551,1],[19308,1],[60741,0]],138727],["wa30",9,[[0,0]],0],["vc33",7,[[8743,1],[8900,1],[25099,1],[50084,1],[52552,1],[18822,0]],164200],["lf52",10,[[0,0]],0],["r11l",6,[[30878,1],[69611,1],[19282,1],[21578,1],[5594,0]],146943],["sc25",6,[[22990,1],[7030,1],[10106,1],[44556,0]],84682],["sp80",6,[[12526,1],[23467,1],[109078,0]],145071],["ar25",8,[[5205,1],[7844,1],[0,0]],13049],["sb26",8,[[2932,1],[4371,0]],7303],["cd82",6,[[32582,1],[12070,1],[67682,1],[12007,1],[6027,1],[3329,1]],133697],["re86",8,[[0,0]],0],["s5i5",8,[[0,0]],0],["ls20",7,[[24631,1],[57486,0]],82117],["ft09",6,[[8302,1],[6715,1],[6102,1],[9047,1],[16006,1],[12404,1]],58576],["su15",9,[[0,0]],0],["tr87",6,[[15756,1],[12960,1],[4208,1],[2206,1],[47000,1],[33141,0]],115271],["g50t",7,[[52104,0]],52104]],"b28_ctl_fb25":[["sk48",8,[[76107,0]],76107],["tn36",7,[[27612,1],[69759,1],[7523,1],[9568,0]],114462],["m0r0",6,[[19194,1],[29512,1],[37101,1],[18875,1],[20864,0]],125546],["bp35",9,[[49297,1],[14133,0]],63430],["cn04",6,[[11603,1],[99204,0]],110807],["dc22",6,[[56135,0]],56135],["tu93",9,[[10360,1],[12080,1],[19261,1],[34593,1],[12495,1],[45466,1],[10549,1],[41441,0]],186245],["lp85",8,[[14111,1],[33521,1],[8630,1],[15650,1],[20545,1],[41442,1],[7598,1],[0,0]],141497],["ka59",7,[[26711,1],[19599,1],[140908,0]],187218],["wa30",9,[[1583,0]],1583],["vc33",7,[[8172,1],[8333,1],[39556,1],[60469,1],[39635,1],[27490,1],[10490,0]],194145],["lf52",10,[[0,0]],0],["r11l",6,[[8022,1],[57651,1],[51183,1],[38158,0]],155014],["sc25",6,[[66966,0]],66966],["sp80",6,[[16939,1],[87291,0]],104230],["ar25",8,[[4090,1],[11740,1],[9393,1],[10731,1],[0,0]],35954],["sb26",8,[[3766,1],[5058,1],[1637,1],[3056,1],[5958,1],[2210,1],[3385,1],[6594,0]],31664],["cd82",6,[[47866,1],[4898,1],[35880,1],[20374,1],[10799,1],[3805,1]],123622],["re86",8,[[3430,1],[11284,1],[7730,1],[14095,1],[0,0]],36539],["s5i5",8,[[13492,1],[5959,0]],19451],["ls20",7,[[74448,1],[15087,0]],89535],["ft09",6,[[11951,1],[3418,1],[6857,1],[10523,1],[58368,0]],91117],["su15",9,[[0,0]],0],["tr87",6,[[32059,1],[10093,1],[6175,1],[10976,1],[41376,0]],100679],["g50t",7,[[50867,0]],50867]],"b32_ctl_r2":[["sk48",8,[[73246,0]],73246],["tn36",7,[[12214,1],[85561,0]],97775],["m0r0",6,[[79487,1],[30582,0]],110069],["bp35",9,[[64712,0]],64712],["cn04",6,[[7570,1],[107604,0]],115174],["dc22",6,[[37195,1],[22820,1],[26645,1],[29818,0]],116478],["tu93",9,[[10462,1],[28599,1],[19878,1],[69073,1],[21850,1],[35290,1],[24906,1],[20572,0]],230630],["lp85",8,[[14811,1],[17215,1],[12545,1],[17378,1],[11659,1],[23862,1],[7810,0]],105280],["ka59",7,[[53654,1],[32547,1],[15370,1],[44936,0]],146507],["wa30",9,[[17201,1],[50817,0]],68018],["vc33",7,[[6529,1],[7387,1],[33995,1],[70420,0]],118331],["lf52",10,[[0,0]],0],["r11l",6,[[14559,1],[40916,1],[29102,1],[61830,1],[8952,0]],155359],["sc25",6,[[20450,1],[6039,1],[19874,1],[52830,0]],99193],["sp80",6,[[5255,1],[128179,0]],133434],["ar25",8,[[5176,1],[29533,1],[12096,1],[8096,1],[0,0]],54901],["sb26",8,[[2461,1],[13112,1],[4017,1],[2246,1],[2509,1],[2106,1],[4514,1],[5708,1]],36673],["cd82",6,[[46128,1],[39366,1],[80796,0]],166290],["re86",8,[[6122,1],[19371,1],[7403,1],[6302,0]],39198],["s5i5",8,[[14362,1],[28138,0]],42500],["ls20",7,[[15281,1],[86660,0]],101941],["ft09",6,[[7722,1],[4816,1],[7137,1],[13741,1],[29251,0]],62667],["su15",9,[[9746,1],[2645,0]],12391],["tr87",6,[[23122,1],[16946,1],[10630,1],[3749,1],[10025,1],[26668,1]],91140],["g50t",7,[[52558,1],[16178,1],[26589,0]],95325]],"b32_ctl_r3":[["sk48",8,[[69454,0]],69454],["tn36",7,[[38446,1],[66270,0]],104716],["m0r0",6,[[71308,1],[17493,1],[39270,1],[47862,1],[22134,0]],198067],["bp35",9,[[55889,0]],55889],["cn04",6,[[15150,1],[35728,1],[52238,1],[40623,1],[2205,0]],145944],["dc22",6,[[58318,0]],58318],["tu93",9,[[7823,1],[16770,1],[12642,1],[24488,1],[14734,1],[32900,1],[8169,1],[21696,0]],139222],["lp85",8,[[9749,1],[26131,1],[14687,1],[32896,1],[21551,1],[18667,1],[8645,1],[8752,0]],141078],["ka59",7,[[25587,1],[54894,1],[6433,0]],86914],["wa30",9,[[39609,1],[19424,0]],59033],["vc33",7,[[6969,1],[7129,1],[18900,1],[41564,1],[30362,1],[14032,1],[22425,1]],141381],["lf52",10,[[0,0]],0],["r11l",6,[[40826,1],[72737,1],[0,0]],113563],["sc25",6,[[47516,1],[5856,1],[45053,1],[17479,1],[15687,0]],131591],["sp80",6,[[4566,1],[124264,0]],128830],["ar25",8,[[8880,1],[7380,1],[16687,1],[0,0]],32947],["sb26",8,[[1537,1],[32130,0]],33667],["cd82",6,[[39821,1],[4497,1],[57774,1],[6464,1],[7304,1],[7925,1]],123785],["re86",8,[[10046,1],[11265,1],[11403,1],[0,0]],32714],["s5i5",8,[[8338,1],[9474,0]],17812],["ls20",7,[[45670,1],[28431,0]],74101],["ft09",6,[[5925,1],[20908,1],[16064,1],[11446,1],[60673,0]],115016],["su15",9,[[0,0]],0],["tr87",6,[[27484,1],[4585,1],[3644,1],[2844,1],[12433,1],[9903,0]],60893],["g50t",7,[[68220,1],[13897,0]],82117]]}
OBSERVED = {"b12_fb25": 106, "b28_ctl_fb25": 95, "b32_ctl_r2": 103, "b32_ctl_r3": 107}

In [ ]:
from math import erfc, sqrt
ERFC = np.vectorize(erfc); INV = np.vectorize(NormalDist().inv_cdf)
def S0(c, mu, s): return 0.5 * ERFC((np.log(np.maximum(c, 1)) - mu) / (s * sqrt(2)))
clr = np.array([t for r in GAMES for _, n, att, _ in GAMES[r] for t, ok in att if ok], float)
opn = np.array([att[-1][0] for r in GAMES for _, n, att, _ in GAMES[r] if att and not att[-1][1]], float)
best = (-np.inf,)
for mu in np.log(np.linspace(5e3, 60e3, 56)):     # grid search for the maximum likelihood (numpy only)
    for s in np.linspace(0.6, 1.8, 25):
        z = (np.log(clr) - mu) / s
        base = (-0.5 * z * z - np.log(s * clr * sqrt(2 * np.pi))).sum(); so = S0(opn, mu, s)
        for q in np.linspace(0, 0.3, 31):
            ll = base + len(clr) * np.log(1 - q) + np.log(q + (1 - q) * so).sum()
            if ll > best[0]: best = (ll, q, mu, s)
_, q, mu, s = best
print(f'fit on the first halves: q {q:.2f}, median {math.exp(mu) / 1e3:.1f}k tokens, s {s:.2f}')
rng = np.random.default_rng(1); N = 2000
def waits(c, n):                                   # tokens still needed to crack, given c tokens already spent (inf = wall)
    sc = float(S0(c, mu, s)) if c > 0 else 1.0
    wall = rng.random(n) < q / (q + (1 - q) * sc)
    z = INV(1 - rng.random(n) * sc)                # lognormal draw above c (inverse CDF)
    return np.where(wall, np.inf, np.exp(mu + s * z) - c)
pred = []
for r in GAMES:
    done_all = [sum(ok for _, ok in att) >= n for _, n, att, _ in GAMES[r]]
    tot_used = sum(u for *_, u in GAMES[r]); open_used = sum(u for (*_, u), d in zip(GAMES[r], done_all) if not d)
    ratio = tot_used / max(1, open_used); total = 0.0
    for (name, n, att, used), d in zip(GAMES[r], done_all):
        k = sum(ok for _, ok in att)
        if d: total += n; continue
        left = np.full(N, used * ratio); lv = np.full(N, k, float); alive = np.ones(N, bool)
        w = waits(att[-1][0] if att and not att[-1][1] else 0.0, N)
        for _ in range(k, n):
            ok = alive & (w <= left); left = np.where(ok, left - w, left); lv += ok; alive = ok
            if not alive.any(): break
            w = waits(0.0, N)
        total += lv.mean()
    pred.append(total)
    print(f'{r}: predicted {total:.1f} levels, observed {OBSERVED[r]}')
print(f'mean: predicted {np.mean(pred):.1f}, observed {np.mean(list(OBSERVED.values())):.2f}')